In [1]:
import time
import asyncpg
from app.data.dto.main.SeaPort import SeaPortDB

db_name="BunkeringBot"
db_user="def_user"
db_password="super_password"
db_host="77.37.96.222"
db_port="5432"

connection_pool = await asyncpg.create_pool(
            database=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port,
            min_size=1,
            max_size=20
        )

In [29]:
import pandas as pd
import asyncio

async def fetch_user_routes_stats(connection_pool):
    async with connection_pool.acquire() as conn:
        # query SPB <-> UAE stats per user
        rows = await conn.fetch(
            """
WITH piter_ports AS (
    SELECT id
    FROM public.ports_vector_new
    WHERE locode IN ('RULED', 'RUSPP', 'RULOM', 'RUBNK', 'RUBRK', 'RUKDT', 'RUGSA', 'RUSZZ')
),
uae_ports AS (
    SELECT id
    FROM public.ports_vector_new
    WHERE left(locode, 2) = 'AE'
)
SELECT
    u.id AS user_id,
    u.email,
    u.telegram_user_name,
    COUNT(*) FILTER (
        WHERE r.departure_port_id IN (SELECT id FROM piter_ports)
          AND r.destination_port_id IN (SELECT id FROM uae_ports)
    ) AS spb_to_uae,
    COUNT(*) FILTER (
        WHERE r.departure_port_id IN (SELECT id FROM uae_ports)
          AND r.destination_port_id IN (SELECT id FROM piter_ports)
    ) AS uae_to_spb,
    COUNT(*) AS total_routes
FROM public.routes r
JOIN public.users u ON u.id = r.user_id
WHERE
    (r.departure_port_id IN (SELECT id FROM piter_ports)
     AND r.destination_port_id IN (SELECT id FROM uae_ports))
    OR
    (r.departure_port_id IN (SELECT id FROM uae_ports)
     AND r.destination_port_id IN (SELECT id FROM piter_ports))
GROUP BY u.id, u.email, u.telegram_user_name
ORDER BY total_routes DESC;
            """
        )

        # convert to pandas DataFrame
        df = pd.DataFrame([dict(r) for r in rows])

        # optional: calculate % split per user
        df['spb_to_uae_pct'] = (df['spb_to_uae'] / df['total_routes'] * 100).round(1)
        df['uae_to_spb_pct'] = (df['uae_to_spb'] / df['total_routes'] * 100).round(1)

        # reorder columns nicely
        df = df[['user_id', 'email', 'telegram_user_name', 'spb_to_uae', 'uae_to_spb', 'spb_to_uae_pct', 'uae_to_spb_pct', 'total_routes']]

        return df

# usage in async context
stats_df = await fetch_user_routes_stats(connection_pool)
stats_df

,user_id,email,telegram_user_name,spb_to_uae,uae_to_spb,spb_to_uae_pct,uae_to_spb_pct,total_routes
0,a9a13e49-d7bc-4bd6-8b7a-c523c3e60e75,precompiledasset@gmail.com,semyon_spb,1,0,100.0,0.0,1
1,ecacbdef-f043-49f0-a6d3-8f2f22e406ea,genbunker@gmail.com,gee_vo,0,1,0.0,100.0,1


In [32]:
import pandas as pd
import asyncio

async def build_full_user_routes_stats(connection_pool):
    # 1️⃣ Fetch all users
    async with connection_pool.acquire() as conn:
        users_rows = await conn.fetch(
            """
            SELECT id AS user_id, email, telegram_user_name
            FROM public.users;
            """
        )
    users_df = pd.DataFrame([dict(r) for r in users_rows])

    # 2️⃣ Fetch all ports
    async with connection_pool.acquire() as conn:
        ports_rows = await conn.fetch(
            """
            SELECT id AS port_id, locode, port_name
            FROM public.ports_vector_new;
            """
        )
    port_map = {str(r['port_id']): f"{r['locode']} - {r['port_name']}" for r in ports_rows}

    # 3️⃣ Fetch all routes
    async with connection_pool.acquire() as conn:
        routes_rows = await conn.fetch(
            """
            SELECT
                user_id,
                departure_port_id,
                destination_port_id
            FROM public.routes;
            """
        )
    routes_df = pd.DataFrame([dict(r) for r in routes_rows])

    # 4️⃣ Handle empty routes
    if routes_df.empty:
        routes_df = pd.DataFrame(columns=['user_id', 'departure_port_id', 'destination_port_id'])

    # 5️⃣ Convert UUIDs to strings for comparison
    routes_df['departure_port_id'] = routes_df['departure_port_id'].astype(str)
    routes_df['destination_port_id'] = routes_df['destination_port_id'].astype(str)

    # 6️⃣ Normalize port order so A->B same as B->A
    routes_df['port1_id'] = routes_df[['departure_port_id', 'destination_port_id']].min(axis=1)
    routes_df['port2_id'] = routes_df[['departure_port_id', 'destination_port_id']].max(axis=1)

    # 7️⃣ Human-readable route
    routes_df['route'] = routes_df['port1_id'].map(port_map) + " ↔ " + routes_df['port2_id'].map(port_map)

    # 8️⃣ Count routes per user per route
    route_counts = (
        routes_df.groupby(['user_id', 'route'])
        .size()
        .reset_index(name='routes_count')
    )

    # 9️⃣ Pivot: users as rows, port pairs as columns
    stats_df = route_counts.pivot_table(
        index='user_id',
        columns='route',
        values='routes_count',
        fill_value=0
    ).reset_index()

    # 10️⃣ Merge with all users to include users without routes
    stats_df = users_df.merge(stats_df, on='user_id', how='left').fillna(0)

    # 11️⃣ Add total_routes column
    route_columns = [c for c in stats_df.columns if c not in ['user_id', 'email', 'telegram_user_name']]
    stats_df['total_routes'] = stats_df[route_columns].sum(axis=1)

    # 12️⃣ Sort by total_routes descending
    stats_df = stats_df.sort_values('total_routes', ascending=False)

    return stats_df

# Usage:
stats_df = await build_full_user_routes_stats(connection_pool)
stats_df.head()

,user_id,email,telegram_user_name,AEFJR - Al Fujayrah ↔ SGSIN - Singapore,AEHZP - Hamriya Free Zone Port ↔ NLAMS - Amsterdam,AEJEA - Jebel Ali Free Zone ↔ INMUN - Mundra,AEJEA - Jebel Ali Free Zone ↔ RUGDZ - Gelendzik,AEJEA - Jebel Ali Free Zone ↔ ZACPT - Cape Town,BGBOJ - Burgas ↔ EGDAM - Dumyat (Damietta),CIABJ - Abidjan ↔ SGSIN - Singapore,...,SEPIT - Piteå ↔ ZACPT - Cape Town,SGSIN - Singapore ↔ CNSHA - Shanghai,SGTPG - Tanjong Pagar ↔ CNSHA - Shanghai,TTPOS - Port-of-Spain ↔ BRSSZ - Santos,TTPOS - Port-of-Spain ↔ CNSHA - Shanghai,VEBOC - Boca Grande ↔ RUVVO - Vladivostok,VEPBL - Puerto Cabello ↔ RULED - Saint Petersburg (ex Leningrad),VEPBL - Puerto Cabello ↔ USNYC - New York,YEADE - Aden ↔ INMUN - Mundra,total_routes
10,a9a13e49-d7bc-4bd6-8b7a-c523c3e60e75,precompiledasset@gmail.com,semyon_spb,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,4.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,37.0
35,ecacbdef-f043-49f0-a6d3-8f2f22e406ea,genbunker@gmail.com,gee_vo,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,28.0
0,86ab9689-9ffb-4db5-94e3-3683757680d1,nik@additivka.ru,VostNik,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0
41,2f6406ec-b0d2-4ac5-89bb-d8779d213b59,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
16,eca5d345-5b55-42a2-8dd3-2638a830dff9,0,srd_jan,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0


In [38]:
import pandas as pd

async def fetch_user_route_counts(connection_pool):
    async with connection_pool.acquire() as conn:
        rows = await conn.fetch(
            """
            SELECT
                u.id AS user_id,
                u.email,
                u.telegram_user_name,
                COUNT(r.id) AS total_routes
            FROM public.routes r
            JOIN public.users u ON r.user_id = u.id
            GROUP BY u.id, u.email, u.telegram_user_name
            ORDER BY total_routes DESC;
            """
        )
    df = pd.DataFrame([dict(r) for r in rows])
    return df

# Usage:
df = await fetch_user_route_counts(connection_pool)
df

,user_id,email,telegram_user_name,total_routes
0,a9a13e49-d7bc-4bd6-8b7a-c523c3e60e75,precompiledasset@gmail.com,semyon_spb,84
1,ecacbdef-f043-49f0-a6d3-8f2f22e406ea,genbunker@gmail.com,gee_vo,41
2,f4dffe09-de16-4d69-b425-d194165f39bc,None,EkCaviar,5
3,2f6406ec-b0d2-4ac5-89bb-d8779d213b59,None,None,3
4,da588390-2709-4c61-bc47-f1e78bfaa5af,None,River_Sea_TN,3
5,eca5d345-5b55-42a2-8dd3-2638a830dff9,None,srd_jan,3
6,86ab9689-9ffb-4db5-94e3-3683757680d1,nik@additivka.ru,VostNik,3
7,db165e99-e5a2-4121-b519-dab6c7a071a6,None,dnitems,2
8,9c431533-2edb-449d-abe6-a5de8220e7b4,Farid@elboil.com,None,2
9,7595ea5b-8af9-408c-b135-a3c513c27bb2,None,None,2


In [47]:
import pandas as pd
import asyncio

async def build_user_routes_df(connection_pool):
    # 1️⃣ Fetch all users
    async with connection_pool.acquire() as conn:
        users_rows = await conn.fetch(
            "SELECT id AS user_id, email, telegram_user_name FROM public.users;"
        )
    users_df = pd.DataFrame([dict(r) for r in users_rows])

    # 2️⃣ Fetch all ports
    async with connection_pool.acquire() as conn:
        ports_rows = await conn.fetch(
            "SELECT id AS port_id, locode, port_name FROM public.ports_vector_new;"
        )
    port_map = {str(r['port_id']): f"{r['locode']} - {r['port_name']}" for r in ports_rows}

    # 3️⃣ Fetch all routes
    async with connection_pool.acquire() as conn:
        routes_rows = await conn.fetch(
            "SELECT user_id, departure_port_id, destination_port_id FROM public.routes;"
        )
    routes_df = pd.DataFrame([dict(r) for r in routes_rows])

    # 4️⃣ Handle empty routes
    if routes_df.empty:
        routes_df = pd.DataFrame(columns=['user_id', 'departure_port_id', 'destination_port_id'])

    # 5️⃣ Convert UUIDs to string for mapping
    routes_df['departure_port_id'] = routes_df['departure_port_id'].astype(str)
    routes_df['destination_port_id'] = routes_df['destination_port_id'].astype(str)

    # 6️⃣ Normalize port order so A ↔ B is same as B ↔ A
    routes_df['port1_id'] = routes_df[['departure_port_id', 'destination_port_id']].min(axis=1)
    routes_df['port2_id'] = routes_df[['departure_port_id', 'destination_port_id']].max(axis=1)

    # 7️⃣ Human-readable route
    routes_df['route'] = routes_df['port1_id'].map(port_map) + " ↔ " + routes_df['port2_id'].map(port_map)

    # 8️⃣ Count routes per user per route
    route_counts = (
        routes_df.groupby(['user_id', 'route'])
        .size()
        .reset_index(name='routes_count')
    )

    # 9️⃣ Pivot to make port pairs columns
    stats_df = route_counts.pivot_table(
        index='user_id',
        columns='route',
        values='routes_count',
        fill_value=0
    ).reset_index()

    # 🔟 Merge with users to include all users
    stats_df = users_df.merge(stats_df, on='user_id', how='left').fillna(0)

    # 1️⃣1️⃣ Add total_routes
    route_columns = [c for c in stats_df.columns if c not in ['user_id', 'email', 'telegram_user_name']]
    stats_df['total_routes'] = stats_df[route_columns].sum(axis=1)

    # 1️⃣2️⃣ Sort by total_routes descending
    stats_df = stats_df.sort_values('total_routes', ascending=False)

    return stats_df

# Usage in async context:
df = await build_user_routes_df(connection_pool)
df

,user_id,email,telegram_user_name,AEFJR - Al Fujayrah ↔ SGSIN - Singapore,AEHZP - Hamriya Free Zone Port ↔ NLAMS - Amsterdam,AEJEA - Jebel Ali Free Zone ↔ INMUN - Mundra,AEJEA - Jebel Ali Free Zone ↔ RUGDZ - Gelendzik,AEJEA - Jebel Ali Free Zone ↔ ZACPT - Cape Town,BGBOJ - Burgas ↔ EGDAM - Dumyat (Damietta),CIABJ - Abidjan ↔ SGSIN - Singapore,...,SEPIT - Piteå ↔ ZACPT - Cape Town,SGSIN - Singapore ↔ CNSHA - Shanghai,SGTPG - Tanjong Pagar ↔ CNSHA - Shanghai,TTPOS - Port-of-Spain ↔ BRSSZ - Santos,TTPOS - Port-of-Spain ↔ CNSHA - Shanghai,VEBOC - Boca Grande ↔ RUVVO - Vladivostok,VEPBL - Puerto Cabello ↔ RULED - Saint Petersburg (ex Leningrad),VEPBL - Puerto Cabello ↔ USNYC - New York,YEADE - Aden ↔ INMUN - Mundra,total_routes
10,a9a13e49-d7bc-4bd6-8b7a-c523c3e60e75,precompiledasset@gmail.com,semyon_spb,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,4.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,37.0
35,ecacbdef-f043-49f0-a6d3-8f2f22e406ea,genbunker@gmail.com,gee_vo,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,28.0
0,86ab9689-9ffb-4db5-94e3-3683757680d1,nik@additivka.ru,VostNik,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0
41,2f6406ec-b0d2-4ac5-89bb-d8779d213b59,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
16,eca5d345-5b55-42a2-8dd3-2638a830dff9,0,srd_jan,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
9,9c431533-2edb-449d-abe6-a5de8220e7b4,Farid@elboil.com,0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
8,7595ea5b-8af9-408c-b135-a3c513c27bb2,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5,b9a3bfbb-e329-4ecd-8312-d06ce41a1a7d,0,PushkarevY,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
31,314ca309-ef60-49d2-812a-3289e3e47429,0,LiberumLgn,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
40,186c1ac5-e7dd-4673-ab38-e3e7c057b67a,0,Evgenius24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
